In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# 1. Setup Path (Assuming you need to access 'src/metrics.py' or data paths)
# Navigate up from Notebook to the project root: .../Bayesian_DA_Budyko_modeling/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(PROJECT_ROOT)

# Import metrics function (assuming it's in src/metrics.py)
from src.metrics import calculate_nse, calculate_kge

# --- Configuration (Adjust file paths based on where run_simulation.py saves data) ---

# Define the folder where your results (e.g., Q_simulated_df) are saved.
# Assuming results are saved in a 'results' folder at the project root.
RESULTS_FOLDER = os.path.join(PROJECT_ROOT, 'results')

# Define the filename for the saved simulated flow data.
# You MUST check your run_simulation.py script to see what it saved!
SIMULATED_Q_FILE = 'Q_simulated_all_scenarios.csv' 
OBSERVED_Q_FILE = 'Q_observed_usgs.csv' 
SPIN_UP_YEARS = 5
SPIN_UP_MONTHS = SPIN_UP_YEARS * 12

# --- 2. Load Data ---

try:
    # Load the combined simulation results (Q_simulated)
    Q_sim_df = pd.read_csv(os.path.join(RESULTS_FOLDER, SIMULATED_Q_FILE), index_col=0, parse_dates=True)
    
    # Load the original observed streamflow (Q_usgs)
    Q_obs_df = pd.read_csv(os.path.join(RESULTS_FOLDER, OBSERVED_Q_FILE), index_col=0, parse_dates=True)
    
    # The 'run_simulation.py' output showed a summary table, but we need the raw time series
    print(f"Successfully loaded simulation data from {SIMULATED_Q_FILE}")

except FileNotFoundError:
    print(f"FATAL: Could not find results files in {RESULTS_FOLDER}.")
    print("Please check the 'RESULTS_FOLDER' and file names.")
    # Exit or create empty dataframes to prevent further errors
    Q_sim_df = None
    Q_obs_df = None

if Q_sim_df is not None:
    
    # Identify the basins and scenarios
    BASIN_IDS = Q_obs_df.columns.tolist()
    SCENARIOS = ['Q_ke', 'Q_assim_base', 'Q_assim_Budyko']
    
    # --- 3. Re-calculate and Summarize Metrics (Post Spin-up) ---

    summary_metrics = {'Basin': BASIN_IDS}
    
    # Ensure the DataFrames are aligned by time (index) and cut off spin-up
    Q_obs_post_spinup = Q_obs_df.iloc[SPIN_UP_MONTHS:]
    Q_sim_post_spinup = Q_sim_df.iloc[SPIN_UP_MONTHS:]

    for scenario in SCENARIOS:
        nse_list = []
        kge_list = []
        
        for basin in BASIN_IDS:
            # Simulated data columns are named like '06452000_Q_ke'
            sim_col = f"{basin}_{scenario}" 
            
            # Extract the correct time series
            Q_sim = Q_sim_post_spinup[sim_col].values
            Q_obs = Q_obs_post_spinup[basin].values
            
            # Calculate metrics
            nse_list.append(calculate_nse(Q_obs, Q_sim))
            kge_list.append(calculate_kge(Q_obs, Q_sim))

        summary_metrics[f'{scenario}_NSE'] = nse_list
        summary_metrics[f'{scenario}_KGE'] = kge_list

    metrics_df = pd.DataFrame(summary_metrics).set_index('Basin')
    
    # print("\n--- Summary Performance (Recalculated) ---")
    # print(metrics_df)

    # --- 4. Visualization (Hydrographs) ---

    # Select one or two basins for detailed visualization
    # PLOT_BASINS = ['06452000', '02315500'] # One bad, one better (from previous output)
PLOT_BASINS = ['06452000', '13340000', '06447000', '06360500', '06354000', '05057000', '07301500', '06191500', '02315500', '06784000'] 

import proplot as pplt

# --- 4. Limit plotting period ---
start_date = '2013-01-12'
end_date = '2014-12-31'

Q_obs_df = Q_obs_df.loc[start_date:end_date]
Q_sim_df = Q_sim_df.loc[start_date:end_date]

import proplot as pplt

fig, axs = pplt.subplots(
    nrows=10,
    ncols=1,
    sharex=True,
    refwidth=8,
    refheight=3,
    facecolor='white'#, dpi=200
)

for i, (ax, basin_id) in enumerate(zip(axs, PLOT_BASINS)):
    ax.plot(Q_obs_df.index, Q_obs_df[basin_id],color='black', lw=1.5, label='USGS Observed ($Q_{obs}$)')
    ax.plot(Q_sim_df.index, Q_sim_df[f'{basin_id}_Q_ke'] , color='grey', ls='--', lw=1.2, label='Base Model ($Q_{ke}$)')
    ax.plot(Q_sim_df.index, Q_sim_df[f'{basin_id}_Q_assim_base'], color='orange', lw=1.2, label='Base Model')
    ax.plot(Q_sim_df.index, Q_sim_df[f'{basin_id}_Q_assim_Budyko'], color='blue', lw=1.2, label='Budyko Model')

    ax.format(title=f'{basin_id}', grid=True, linewidth=0.6, fontsize=20)
    if i == 0:
        ax.legend(loc='ul', frame=False, ncols=1, fontsize=20)

fig.format(
    xlabel='Date',
    ylabel='Streamflow (mm/day)',
)
pplt.show()


FATAL: Could not find results files in c:\Users\hdagne1\Box\Dr.Mesfin Research\Codes\DA\DA_Github_repo\Bayesian_DA_Budyko_modeling\results.
Please check the 'RESULTS_FOLDER' and file names.


AttributeError: 'NoneType' object has no attribute 'loc'

In [139]:
all_simulated_components = pd.read_csv(r'C:\Users\hdagne1\Box\Dr.Mesfin Research\Codes\DA\DA_Github_repo\Bayesian_DA_Budyko_modeling\results\time_series_by_basin\05057000_simulated_components.csv')

In [142]:
all_simulated_components

,Date,Q_ke,Q_assim_base,Q_assim_Budyko,ET_ke,ET_assim_base,ET_assim_Budyko,S_ke,S_assim_base,S_assim_Budyko,...,Qs_assim_base,Qs_assim_Budyko,Qb_ke,Qb_assim_base,Qb_assim_Budyko,Perc_ke,Perc_assim_base,Perc_assim_Budyko,omega_true,omega_mlr
0,2000-01-01,214.127048,1.310815e-02,1.256172e-02,3.204635,1.577818,1.845138,50.496652,1.251071,0.775827,...,0.000000,0.000000,28.691025,1.310815e-02,1.256172e-02,2.657719,0.00000,0.00000,5.102633,6.589741
1,2000-02-01,66.224857,1.662655e-02,1.331710e-02,11.425029,8.619303,9.065066,7.963382,1.172336,0.253237,...,0.000000,0.000000,26.994711,1.662655e-02,1.331710e-02,0.419125,0.05864,0.01259,10.000000,7.156457
2,2000-03-01,38.050852,7.245073e+00,6.509555e+00,13.168956,19.053734,18.134635,0.000000,0.000000,0.000000,...,7.242987,6.507708,25.375028,2.085749e-03,1.847567e-03,0.000000,0.00000,0.00000,10.000000,6.945813
3,2000-04-01,37.930626,1.407840e+01,1.407836e+01,13.519525,27.597624,27.597624,0.000000,0.000000,0.000000,...,14.078099,14.078099,23.852527,2.995136e-04,2.653106e-04,0.000000,0.00000,0.00000,10.000000,6.860674
4,2000-05-01,62.747367,4.032603e+01,4.032603e+01,20.081498,60.407490,60.407490,0.000000,0.000000,0.000000,...,40.325992,40.325992,22.421375,4.301016e-05,3.809861e-05,0.000000,0.00000,0.00000,6.550794,6.060116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,2014-08-01,51.222502,5.122078e+01,5.122057e+01,22.805144,74.025719,74.025719,0.000000,0.000000,0.000000,...,51.220575,51.220575,0.001928,2.028067e-04,2.803716e-10,0.000000,0.00000,0.00000,6.164256,5.120200
176,2014-09-01,12.192724,1.219094e+01,1.219091e+01,13.047728,25.238640,25.238640,0.000000,0.000000,0.000000,...,12.190912,12.190912,0.001812,2.912304e-05,4.026136e-11,0.000000,0.00000,0.00000,6.512918,5.930920
177,2014-10-01,1.403413,1.401714e+00,1.401710e+00,10.350427,11.752137,11.752137,0.000000,0.000000,0.000000,...,1.401710,1.401710,0.001703,4.182069e-06,5.781531e-12,0.000000,0.00000,0.00000,6.349892,6.388364
178,2014-11-01,0.001601,6.005451e-07,8.302279e-13,4.431014,4.431014,4.431014,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.001601,6.005451e-07,8.302279e-13,0.000000,0.00000,0.00000,5.420110,7.122416
